In [0]:
import requests
import json
import pandas as pd
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS workspace")
# spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.cricket_api_project")

base_path = '/Volumes/workspace/default/cricket_api_project'


### calling cricket API

In [0]:
import requests
import json

API_KEY = "YOUR_API_KEY"

# api_url = f"https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}&offset=0"
api_url = "https://api.cricapi.com/v1/currentMatches?apikey=d9efd47b-a102-4d25-a562-20cc1cb18e66&offset=0"

response = requests.get(api_url)
response.raise_for_status()

api_data = response.json()

print(api_data.keys())


In [0]:
print(json.dumps(api_data, indent=4)[:2000])

#### SAVE RAW API Response in the Volumns

In [0]:
raw_file_path = f"{base_path}/current_matches_raw.json"

with open (raw_file_path, 'w') as f:
  json.dump(api_data, f)

print(f"Raws API data is sav at the following path: {raw_file_path}")

In [0]:

bronze_data = [{
    "source_api":api_url,
    "raw_json":json.dumps(api_data),
    "ingestion_time":None
}]

bronze_schema = StructType([
    StructField("source_api", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_time", TimestampType(), True)
])


bronze_df = spark.createDataFrame(bronze_data, schema=bronze_schema)\
    .withColumn("ingestion_time",current_timestamp())

display(bronze_df)

In [0]:
bronze_data

In [0]:
bronze_schema

In [0]:
bronze_df.write.format('delta').mode("overwrite").saveAsTable("workspace.default.cricket_bronze_current_matches")
print("Data saved to table cricket_bronze_current_matches")